# Modelización — Detección de Deepfakes (Fase 2)

**TFM — Máster en Ciencia de Datos e IA (UCM)**

Pipeline de modelización sobre los rostros ya extraídos (Fase 1):

1. **Embeddings** — la CNN preentrenada y congelada (EfficientNet) convierte cada
   rostro en un vector. Se calculan una vez y se **cachean**.
2. **Baseline** — clasificador sobre la media de embeddings (sin tiempo).
3. **Híbrido CNN+LSTM** — modela la coherencia temporal del clip.
4. **AutoML** — referencia automática sobre embeddings promediados, para demostrar
   que el híbrido **aporta más que un AutoML**.
5. **Cross-manipulation** — generalización a un método de manipulación NO visto.
6. **Lectura de negocio** — umbral de decisión por coste FP/FN y matrices de confusión.

> Requiere los embeddings calculados. Si aún no existen, la sección 1 los genera a
> partir de `data/interim/` (necesita haber ejecutado la extracción facial).


## Preparación del entorno en Colab

Solo actúa en Colab: monta Drive, clona/actualiza el repo, instala dependencias y fija rutas. **Edita `REPO_URL`**. En local, sáltala.

In [ ]:
# --- Bootstrap para Google Colab (en local no hace nada) ---
import os
from pathlib import Path
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["TFM_WORKSPACE"] = "/content/drive/MyDrive/TFM_Deepfake"

    REPO_URL = "https://github.com/TU_USUARIO/TU_REPO.git"   # <-- EDITA ESTO
    PROJECT_ROOT = "/content/TFM_Deepfake_Detection"
    os.environ["TFM_PROJECT_ROOT"] = PROJECT_ROOT
    if not Path(PROJECT_ROOT).exists():
        !git clone {REPO_URL} {PROJECT_ROOT}
    else:
        !cd {PROJECT_ROOT} && git pull -q
    !pip install -q timm grad-cam gradio pyyaml tqdm
    !pip install -q --no-deps facenet-pytorch
    print("Entorno de Colab listo.")

## 0. Configuración (compatible local / Colab)

In [ ]:
import os, sys
from pathlib import Path
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
PROJECT_ROOT = Path(os.environ.get("TFM_PROJECT_ROOT", "/content/TFM_Deepfake_Detection")) \
    if IN_COLAB else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.utils.seeds import set_seed, load_config, get_device
from src.utils.paths import get_paths, ensure_dirs

cfg = load_config(PROJECT_ROOT / "config" / "config.yaml")
set_seed(cfg["project"]["seed"])
paths = get_paths(cfg, PROJECT_ROOT); ensure_dirs(paths)
DEVICE = get_device()
MAX_LEN = cfg["face_extraction"]["frames_per_video"]
print("Entorno:", "Colab" if IN_COLAB else "local", "| Device:", DEVICE)

## 1. Embeddings (calcular o cargar)

Si el manifiesto ya existe, se carga. Si no, se calcula a partir de los rostros de
`data/interim/` (esto solo se hace una vez; luego queda cacheado en Drive).

In [ ]:
from src.features.embeddings import build_embeddings
from src.data.dataset import enumerate_videos, load_official_splits, assign_splits
from src.data.sequence_dataset import load_manifest

manifest_path = paths["processed"] / "embeddings_manifest.csv"

if manifest_path.exists():
    manifest = load_manifest(paths["processed"])
    print(f"Manifiesto cargado: {len(manifest)} vídeos con embeddings.")
else:
    inv = enumerate_videos(paths["raw"], compression=cfg["dataset"]["compression"],
                           methods=cfg["dataset"]["manipulation_methods"])
    splits_dir = paths["raw"] / "splits"
    if splits_dir.exists():
        inv["split"] = assign_splits(inv, load_official_splits(splits_dir))
    manifest = build_embeddings(
        inv, paths["interim"], paths["processed"],
        backbone=cfg["model"]["backbone"], batch_size=32,
    )

EMBED_DIM = int(manifest["embed_dim"].iloc[0])
print("Dimensión de embedding:", EMBED_DIM)
manifest.head()

## 2. Conjuntos de datos y DataLoaders

Se usa el **split oficial** si el subconjunto descargado lo permite; si deja algún conjunto vacío (habitual con pocos vídeos), se cae automáticamente a un reparto **estratificado** que garantiza train/val/test con ambas clases.

In [ ]:
from src.data.sequence_dataset import (
    SequenceDataset, collate_sequences, get_splits, class_weights,
)

parts = get_splits(manifest)
for k, v in parts.items():
    print(f"{k:5}: {len(v):4d} vídeos | fakes={int((v['label']==1).sum())}"
          f" | reales={int((v['label']==0).sum())}")

BATCH = cfg["training"]["batch_size"]
loaders = {
    name: DataLoader(SequenceDataset(df, MAX_LEN), batch_size=BATCH,
                     shuffle=(name == "train"), collate_fn=collate_sequences)
    for name, df in parts.items()
}
POS_WEIGHT = class_weights(parts["train"])
print("pos_weight (desbalance):", round(POS_WEIGHT, 3))

## 3. Entrenamiento: baseline y modelo híbrido

In [ ]:
from src.models.baseline import FrameMeanBaseline
from src.models.hybrid import CNNLSTM
from src.training.trainer import train_model, _predict_proba

EPOCHS = cfg["training"]["epochs"]
LR = cfg["training"]["learning_rate"]

# --- Baseline (sin tiempo) ---
baseline = FrameMeanBaseline(EMBED_DIM, hidden=cfg["model"]["hidden_size"])
train_model(baseline, loaders["train"], loaders["val"], epochs=EPOCHS, lr=LR,
            pos_weight=POS_WEIGHT, device=DEVICE,
            checkpoint_path=paths["models"] / "baseline.pt")

# --- Híbrido CNN+LSTM ---
hybrid = CNNLSTM(EMBED_DIM, hidden=cfg["model"]["hidden_size"],
                 num_layers=cfg["model"]["num_layers"],
                 rnn_type=cfg["model"]["temporal"],
                 dropout=cfg["model"]["dropout"])
train_model(hybrid, loaders["train"], loaders["val"], epochs=EPOCHS, lr=LR,
            pos_weight=POS_WEIGHT, device=DEVICE,
            checkpoint_path=paths["models"] / "hybrid.pt")

## 4. Referencia AutoML (sobre embeddings promediados)

Una referencia **automática** sobre el vector medio de cada vídeo. Si el híbrido la supera, demostramos que modelar la dimensión temporal aporta valor sobre un enfoque tabular automatizado. (Para una versión más potente, puede usarse FLAML o auto-sklearn.)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

def pooled_matrix(df):
    """Carga embeddings y los promedia por vídeo -> X [N, D], y [N]."""
    X = np.stack([np.load(p).mean(axis=0) for p in df["embedding_path"]])
    return X, df["label"].values.astype(int)

Xtr, ytr = pooled_matrix(parts["train"])
Xte, yte = pooled_matrix(parts["test"])

candidates = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}
# Selección automática del mejor por AUC en validación cruzada
scores = {n: cross_val_score(m, Xtr, ytr, cv=3, scoring="roc_auc").mean()
          for n, m in candidates.items()}
best_name = max(scores, key=scores.get)
print("AUC CV:", {k: round(v, 3) for k, v in scores.items()}, "-> mejor:", best_name)

automl = candidates[best_name].fit(Xtr, ytr)
automl_prob = automl.predict_proba(Xte)[:, 1]

## 5. Evaluación comparativa en test

In [ ]:
from src.evaluation.metrics import compute_metrics, metrics_table

# Probabilidades en test para los modelos de deep learning
y_true, base_prob = _predict_proba(baseline, loaders["test"], DEVICE)
_, hyb_prob = _predict_proba(hybrid, loaders["test"], DEVICE)

results = {
    "Baseline (media+MLP)": compute_metrics(y_true, base_prob),
    f"AutoML ({best_name})": compute_metrics(yte, automl_prob),
    "Híbrido CNN+LSTM": compute_metrics(y_true, hyb_prob),
}
table = metrics_table(results)
display(table)
table.to_csv(paths["figures"] / "tabla_comparativa.csv")
print("Tabla guardada en reports/figures/tabla_comparativa.csv")

## 6. Experimento cross-manipulation (generalización)

Entrenamos con tres métodos y evaluamos sobre el cuarto, **no visto**. Mide si el
modelo aprende artefactos generales del deepfake o solo "memoriza" un método.

In [ ]:
from src.data.sequence_dataset import split_cross_manipulation

cm_cfg = cfg["dataset"]["cross_manipulation"]
cm = split_cross_manipulation(manifest, cm_cfg["train_methods"], cm_cfg["holdout_method"])
print(f"Holdout (no visto en train): {cm_cfg['holdout_method']}")
for k, v in cm.items():
    print(f"{k:5}: {len(v):4d} vídeos")

cm_loaders = {
    name: DataLoader(SequenceDataset(df, MAX_LEN), batch_size=BATCH,
                     shuffle=(name == "train"), collate_fn=collate_sequences)
    for name, df in cm.items()
}
hybrid_cm = CNNLSTM(EMBED_DIM, hidden=cfg["model"]["hidden_size"],
                    rnn_type=cfg["model"]["temporal"], dropout=cfg["model"]["dropout"])
train_model(hybrid_cm, cm_loaders["train"], cm_loaders["val"], epochs=EPOCHS, lr=LR,
            pos_weight=class_weights(cm["train"]), device=DEVICE,
            checkpoint_path=paths["models"] / "hybrid_crossmanip.pt")

y_cm, prob_cm = _predict_proba(hybrid_cm, cm_loaders["test"], DEVICE)
print("\nRendimiento sobre el método NO visto:")
display(metrics_table({f"Híbrido (test={cm_cfg['holdout_method']})":
                       compute_metrics(y_cm, prob_cm)}))

## 7. Lectura de negocio: umbral por coste y matriz de confusión

En el caso KYC, un **falso negativo** (deepfake que pasa el filtro) es mucho más
caro que un **falso positivo** (cliente real enviado a revisión manual). Elegimos
el umbral que minimiza el coste esperado, no el que maximiza el F1.

In [ ]:
from src.evaluation.metrics import choose_threshold_by_cost, plot_confusion, compute_metrics

# Costes RELATIVOS de ejemplo: un FN cuesta 20x un FP (ajústalo al caso real).
COST_FP, COST_FN = 1, 20
opt = choose_threshold_by_cost(y_true, hyb_prob, cost_fp=COST_FP, cost_fn=COST_FN)
print(f"Umbral por defecto: 0.50 | Umbral óptimo por coste: {opt['threshold']:.2f}")

print("\nMétricas del híbrido en ambos umbrales:")
display(metrics_table({
    "Híbrido @0.50": compute_metrics(y_true, hyb_prob, 0.50),
    f"Híbrido @{opt['threshold']:.2f} (coste)": compute_metrics(y_true, hyb_prob, opt["threshold"]),
}))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_confusion(y_true, hyb_prob, 0.50, ax=axes[0], title="Umbral 0.50")
plot_confusion(y_true, hyb_prob, opt["threshold"], ax=axes[1],
               title=f"Umbral {opt['threshold']:.2f} (coste)")
plt.tight_layout()
plt.savefig(paths["figures"] / "matriz_confusion.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. Conclusiones de la fase

Completar con los resultados reales tras la ejecución:

- **¿Aporta la dimensión temporal?** Comparar el híbrido CNN+LSTM frente al baseline
  y al AutoML en la tabla de la sección 5.
- **¿Generaliza?** La caída de rendimiento en cross-manipulation (sección 6) indica
  cuánto depende el modelo del método de manipulación concreto.
- **Decisión de negocio:** el umbral por coste (sección 7) traduce el modelo a una
  política operativa que prioriza no dejar pasar deepfakes.

**Siguiente paso (Fase 3):** explicabilidad con Grad-CAM sobre los frames para
visualizar *por qué* el modelo decide, y traducción a lenguaje de negocio.
